# Agentic RLHF for Aerospace using Browser + DPO
This notebook builds a browser-based dataset and fine-tunes a model using DPO for aerospace tasks.

In [1]:
!pip install -q google-search-results newspaper3k trl datasets transformers accelerate bitsandbytes

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 73.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 4.7 MB/s eta 0:00:00


## Add your SerpAPI Key

In [2]:
SERP_API_KEY = "PASTE_YOUR_SERPAPI_KEY_HERE"

## Browser Search Function

In [3]:
!pip install lxml[html_clean]
from serpapi import GoogleSearch
from newspaper import Article

def browser_search(query, num_results=3):
    params = {
        "engine": "google",
        "q": query,
        "api_key": SERP_API_KEY,
        "num": num_results
    }

    search = GoogleSearch(params)
    results = search.get_dict()

    links = [r['link'] for r in results.get("organic_results", [])[:num_results]]
    extracted_text = ""

    for link in links:
        try:
            article = Article(link)
            article.download()
            article.parse()
            extracted_text += article.text[:1500] + "\n\n"
        except:
            continue

    return extracted_text

## Create Preference Dataset from Browser

In [4]:
import json

def create_preference_example(prompt):
    web_text = browser_search(prompt)

    chosen = f"""Plan: Search web for accurate info → summarize.\n\nFinal Answer:\n{web_text[:800]}"""
    rejected = "This topic has general rules and information available online."

    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected
    }

prompts = [
    "FAA Part 107 BVLOS rules 2025",
    "NASA Artemis mission latest update",
    "EASA drone regulation updates",
    "Composite laminate defects aerospace",
    "UAV pre flight checklist best practices",
    "DO-160 EMI standards avionics",
    "NDT methods for aerospace composites",
    "Aircraft safety statistics 2025 report"
]

dataset = [create_preference_example(p) for p in prompts]

with open("browser_aerospace_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

print("Dataset created!")


Dataset created!


## Load Model and Train with DPO

In [5]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOTrainer, DPOConfig

dataset = load_dataset("json", data_files="browser_aerospace_dataset.json")["train"]

model_name = "Qwen/Qwen1.5-0.5B"

model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

config = DPOConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    logging_steps=1,
    output_dir="dpo_aerospace_agent"
)

trainer = DPOTrainer(
    model=model,
    args=config,
    train_dataset=dataset
)

trainer.train()
trainer.save_model("final_dpo_Aerospace_agent")
print("Training complete!")

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,0.688027
2,0.671799
3,0.677300
4,0.675515


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


In [9]:
# Step 1 — Load Base vs Fine-Tuned Model

!pip install -q transformers bitsandbytes accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

base_model_id = "Qwen/Qwen2-0.5B"
ft_model_path = "/content/final_dpo_Aerospace_agent"  # your saved model

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    quantization_config=quantization_config
)

ft_model = AutoModelForCausalLM.from_pretrained(
    ft_model_path,
    device_map="auto",
    quantization_config=quantization_config
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [10]:
# Step 2 — Define Aerospace Test Prompts

test_prompts = [
    "FAA requirements for UAV airworthiness certification",
    "Composite airframe inspection checklist before flight",
    "BVLOS regulatory requirements for drones",
    "Steps in structural validation of UAV wing",
    "MIL standard for avionics wiring harness in aircraft"
]

In [11]:
# Step 3 — Text Generation Function

def generate(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.7,
            do_sample=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [12]:
# Step 4 — Generate Outputs (Before vs After)

results = []

for p in test_prompts:
    base_out = generate(base_model, p)
    ft_out = generate(ft_model, p)

    results.append({
        "prompt": p,
        "base": base_out,
        "fine_tuned": ft_out
    })

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


In [13]:
# Step 5 — Automatic Scoring (Engineering Quality Metrics)

keywords = [
    "FAA", "EASA", "NASA", "inspection", "checklist",
    "composite", "airworthiness", "structural", "MIL", "standard"
]

def score(text):
    return sum(1 for k in keywords if k.lower() in text.lower())

for r in results:
    r["base_score"] = score(r["base"])
    r["ft_score"] = score(r["fine_tuned"])

In [14]:
# Step 6 — Display Results Clearly (for report)

for r in results:
    print("="*80)
    print("PROMPT:\n", r["prompt"])
    print("\n--- BASE MODEL OUTPUT ---\n")
    print(r["base"][:800])
    print("\nScore:", r["base_score"])

    print("\n--- FINE TUNED MODEL OUTPUT ---\n")
    print(r["fine_tuned"][:800])
    print("\nScore:", r["ft_score"])

PROMPT:
 FAA requirements for UAV airworthiness certification

--- BASE MODEL OUTPUT ---

FAA requirements for UAV airworthiness certification, which are based on the requirements set out in the AIP, are based on the requirements as originally set out in the ICAO Annex 137, which was subsequently superseded by the ICAO Annex 747. These requirements are based on the requirements for aircraft certification outlined in Annex 747, which has been superseded by the ICAO Annex 137.

Score: 2

--- FINE TUNED MODEL OUTPUT ---

FAA requirements for UAV airworthiness certification
Posted on 24.03.2020
A drone’s airworthiness certification is the assurance that the drone belongs to the FAA and is in good operational condition. This is a non-essential certification, but it is a good one. The FAA has established a set of guidelines that are used to evaluate the airworthiness of a drone.
When selecting a drone for a flight, the requirements are:
• The drone must be certified by the FAA.
• The drone m

In [15]:
# Step 7 — Numerical Summary (for GitHub/report)

import numpy as np

base_scores = [r["base_score"] for r in results]
ft_scores = [r["ft_score"] for r in results]

print("----- Numerical Evaluation Summary -----")
print(f"Average Base Model Score: {np.mean(base_scores):.2f}")
print(f"Average Fine-Tuned Model Score: {np.mean(ft_scores):.2f}")
print("----------------------------------------")

----- Numerical Evaluation Summary -----
Average Base Model Score: 2.00
Average Fine-Tuned Model Score: 1.60
----------------------------------------


In [16]:
# Step 8 — Save Results to File

import json

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Saved evaluation_results.json")

Saved evaluation_results.json
